In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import pandas as pd
from sklearn import compose, impute, metrics, model_selection
from sklearn import preprocessing, pipeline, tree
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import skew
from sklearn.metrics import mean_squared_error
from scipy.stats import skew
from sklearn.metrics import make_scorer

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


**1. Define the objective: what are you trying to predict?**
The goal is to predict the final sale price (SalePrice) of residential homes based on various descriptive features provided in the dataset, such as building type, neighborhood, square footage, and year built.

**2. How should you frame this problem as a machine learning problem?**
This is a supervised regression problem, since the variable to be predicted (SalePrice) is continuous and numeric. The model will learn a mapping between input features (house attributes) and the continuous output (sale price) using labeled examples from the training data.

**3. How should performance be measured?**
Performance should be measured using Root Mean Squared Logarithmic Error (RMSLE).
The Kaggle Competition uses it as a metric &
RMSLE penalizes large relative errors more than small absolute ones, which is suitable for house prices because their scale varies widely. It focuses on percentage differences rather than raw price differences, ensuring that overestimates and underestimates are treated symmetrically on a log scale.

In [ ]:
# Define RMSLE metric (RMSE on log scale)
def rmse_log(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(y_true_log, y_pred_log))


scorer = make_scorer(lambda yt, yp: -rmse_log(yt, yp))  # for cross_val_score

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

**Plan:**
1. Load the data.
The dataset is loaded from the Kaggle House Prices competition. The training set contains both input features and the target variable (SalePrice).

2. Check the size and type of data.
The training dataset consists of 1460 observations and 81 columns, where 80 are explanatory features and 1 is the target variable.
Data types include:

Numeric features: e.g., LotArea, GrLivArea, YearBuilt

Categorical features: e.g., Neighborhood, HouseStyle, Exterior1st

Ordinal features: e.g., OverallQual, ExterCond, BsmtQual

Preprocessing will require handling both numeric and categorical attributes appropriately.

In [ ]:
import pandas as pd

train_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
        index_col="Id"
    )
)

data = train_df
print("Loaded data shape:", data.shape)
data.head()

# --- Numeric vs categorical features ---
numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = data.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


Plan:


**1 Discuss the types of features.The dataset contains a mix of:**


Categorical variables: qualitative labels such as Neighborhood, MSZoning, and SaleCondition.

Ordinal variables: quality or condition ratings like OverallQual, ExterQual, and KitchenQual, which have inherent order but not consistent spacing.

Numeric variables: continuous or discrete measurements such as LotArea, GrLivArea, 1stFlrSF, and GarageCars.
Most numeric features are bounded, while some categorical features have limited distinct levels.

**2 What features have missing values?**
Several features contain missing values. Notably, variables related to the basement, garage, and pool often have missing entries because not all homes include these attributes. Examples include:

GarageType, GarageYrBlt, GarageFinish

BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1

PoolQC, Fence, and MiscFeature

Some missing values signify absence of the feature rather than missing data (e.g., no garage or pool), so they should be handled accordingly during preprocessing.

**3 Are there any outliers? How can you tell?**
Yes, there are outliers—particularly in variables like GrLivArea and LotArea.
Scatterplots of GrLivArea versus SalePrice reveal a few houses with extremely large living areas but relatively low prices, which deviate from the main trend. These outliers may distort the regression model and could be removed or capped based on domain reasoning.

**4 What features appear useful for the task? Which appear useless? Why?**
Useful features likely include:

OverallQual and GrLivArea — strong positive correlation with SalePrice.

TotalBsmtSF, GarageCars, and YearBuilt — known to influence home values.

Less useful or redundant features include:

Id (purely an identifier).

Features with many missing values and weak correlation with price, such as PoolQC or MiscFeature.

**5 Identify and characterize the target distribution.**
SalePrice is right-skewed — most homes are moderately priced with a small number of expensive houses.
Applying a logarithmic transformation helps normalize the distribution, making it more symmetric and suitable for linear modeling techniques that assume normal residuals.

**6 Visualize the data.**
Histograms and boxplots show:

SalePrice and GrLivArea are positively correlated.

Qualitative features like OverallQual show clear stepwise increases in median sale price.

**7 Study correlations between attributes.**
The correlation matrix shows that:

OverallQual and GrLivArea have the highest correlation with SalePrice.

Other correlated features include GarageCars, TotalBsmtSF, and YearBuilt.

Some variables, such as 1stFlrSF and TotalBsmtSF, are mutually correlated, suggesting potential multicollinearity that should be addressed in feature selection.

In [ ]:
# 3. Explore the Data

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Display dataset information
print("Dataset shape:", data.shape)
print("\nData types summary:")
print(data.dtypes.value_counts())

# --- Basic info ---
print("\nSample overview:")
display(data.head())

# --- Check missing values ---
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nNumber of features with missing values: {len(missing)}")

plt.figure(figsize=(10,6))
sns.barplot(x=missing.values, y=missing.index, color="salmon")
plt.title("Features with Missing Values")
plt.xlabel("Number of Missing Entries")
plt.ylabel("Feature")
plt.show()

# --- Numeric vs categorical features ---
numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = data.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
warnings.filterwarnings("ignore", message="The figure layout has changed to tight")

# --- Check target variable ---
plt.figure(figsize=(6,4))
sns.histplot(data["SalePrice"], kde=True)
plt.title("Distribution of Target Variable (SalePrice)")
plt.xlabel("SalePrice")
plt.show()

print("Skewness of SalePrice:", data["SalePrice"].skew())

# Skewed → motivates log transformation in feature engineering.

# --- Correlation with target ---
corr = data.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
plt.figure(figsize=(8,6))
sns.barplot(x=corr.values[:20], y=corr.index[:20], color="skyblue")
plt.title("Top 20 Features Most Correlated with SalePrice")
plt.xlabel("Correlation")
plt.show()

print("\nHighly correlated features with SalePrice:")
print(corr.head(10))

# --- Example scatter plots for some strong numeric features ---
top_corr_features = corr.head(5).index.tolist()[1:]  # excluding SalePrice itself
sns.pairplot(data[top_corr_features + ["SalePrice"]], diag_kind="kde")
plt.suptitle("Pairwise Relationships of Strong Predictors", y=1.02)
plt.show()

# --- Outlier detection in key variables ---
plt.figure(figsize=(10,6))
sns.boxplot(x=data["OverallQual"], y=data["SalePrice"])
plt.title("SalePrice vs OverallQual (Possible Outliers)")
plt.show()

plt.figure(figsize=(10,6))
sns.scatterplot(x=data["GrLivArea"], y=data["SalePrice"])
plt.title("GrLivArea vs SalePrice (Outliers at High GrLivArea)")
plt.show()


print("""
Observations:
1. SalePrice is right-skewed → log transformation justified.
2. Several numeric predictors (e.g., OverallQual, GrLivArea, GarageCars, TotalBsmtSF)
   are strongly correlated with SalePrice — likely useful features.
3. Some variables have many missing values (e.g., PoolQC, MiscFeature, Alley, Fence)
   which may need to be dropped or encoded as 'None'.
4. Outliers detected in GrLivArea and SalePrice → may need filtering or transformation.
5. Many categorical variables (Neighborhood, Exterior, etc.) → label or one-hot encoding required.
These findings guide to an extend the cleaning and feature engineering in Section 4.
""")


# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
train_features_df = data.drop("SalePrice", axis=1)
train_target = data.loc[:, "SalePrice"]

In [ ]:
# Helper function

def ensure_dataframe(X, columns=None):
    """Convert numpy array to pandas DataFrame if needed."""
    if isinstance(X, pd.DataFrame):
        return X.copy()
    elif isinstance(X, np.ndarray):
        if columns is None:
            columns = [f'col_{i}' for i in range(X.shape[1])]
        return pd.DataFrame(X, columns=columns)
    else:
        raise ValueError("Input must be a pandas DataFrame or numpy ndarray.")


# Custom Transformers
class YearFixer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = ensure_dataframe(X)
        if 'GarageYrBlt' in X.columns and 'YearBuilt' in X.columns:
            # Create a mask that handles NaN values properly
            mask = (X['GarageYrBlt'].notna()) & (X['GarageYrBlt'].fillna(0) > 2018)
            X.loc[mask, 'GarageYrBlt'] = X.loc[mask, 'YearBuilt']
        return X


class LotFrontageMedianImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = ensure_dataframe(X)
        if 'LotFrontage' in X.columns:
            self.median_ = X['LotFrontage'].median()
        else:
            self.median_ = None
        return self
    def transform(self, X):
        X = ensure_dataframe(X)
        if 'LotFrontage' in X.columns and self.median_ is not None:
            X['LotFrontage'] = X['LotFrontage'].fillna(self.median_)
        return X


class QualityMapper(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.mapping = {'Ex': 9, 'Gd': 7, 'TA': 5, 'Fa': 3, 'Po': 2}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = ensure_dataframe(X).copy()
        for c in self.cols:
            if c in X.columns:
                mapped = X[c].map(self.mapping)
                X[c] = mapped.where(mapped.notna(), X[c])
        return X


class NeighborhoodModeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, neighborhood_col='Neighborhood', features=None):
        self.neighborhood_col = neighborhood_col
        self.features = features or []
    def fit(self, X, y=None):
        X = ensure_dataframe(X)
        grouped = (
            X.groupby(self.neighborhood_col)[self.features]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        )
        self.fill_map_ = grouped.to_dict(orient='index')
        return self
    def transform(self, X):
        X = ensure_dataframe(X)
        for f in self.features:
            X[f] = X.apply(
                lambda row: self.fill_map_.get(row[self.neighborhood_col], {}).get(f, row[f]),
                axis=1
            )
        return X


class SkewedFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, skew_threshold=0.75):
        self.skew_threshold = skew_threshold
    
    def fit(self, X, y=None):
        X = ensure_dataframe(X)
        # Handle NaN values before calculating skewness
        X_clean = X.fillna(0)  # or use median imputation
        skewness = X_clean.apply(lambda x: skew(x))
        self.skewed_features_ = skewness[skewness > self.skew_threshold].index
        return self
    
    def transform(self, X):
        X = ensure_dataframe(X)
        for f in self.skewed_features_:
            if f in X.columns:
                # Ensure no negative values before log transform
                min_val = X[f].min()
                if min_val <= 0:
                    X[f] = X[f] - min_val + 1  # shift to positive values
                X[f] = np.log1p(X[f])
        return X


class RareCategoryMerger(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.05):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        X = ensure_dataframe(X)
        self.replace_map_ = {}
        for col in X.select_dtypes(include='object').columns:
            freq = X[col].value_counts(normalize=True)
            rare = freq[freq < self.threshold].index
            if len(rare) > 0:
                most_common = freq.index[0]
                self.replace_map_[col] = {r: most_common for r in rare}
        return self
    
    def transform(self, X):
        X = ensure_dataframe(X)
        for col, mapping in self.replace_map_.items():
            if col in X.columns:
                # Use map instead of replace to avoid downcasting warnings
                X[col] = X[col].map(lambda x: mapping.get(x, x))
        return X


# Pipelines
grades = [
    'OverallQual', 'OverallCond', 'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual',
    'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]

neighborhood_feats = [
    'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd',
    'Electrical', 'Functional', 'SaleType'
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('skewfix', SkewedFeatureTransformer()),
    ('scaler', StandardScaler())
])
numeric_pipeline.steps[0][1].set_output(transform="pandas") # Set imputer to output pandas

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('raremerge', RareCategoryMerger(threshold=0.05)),
    ('encoder', OneHotEncoder(handle_unknown='infrequent_if_exist', sparse_output=False, ))
])
categorical_pipeline.steps[0][1].set_output(transform="pandas") # Set imputer to output pandas

preprocessor = Pipeline([
    ('fix_years', YearFixer()),
    ('fix_lotfront', LotFrontageMedianImputer()),
    ('map_quality', QualityMapper(cols=grades)),
    ('neigh_mode_impute', NeighborhoodModeImputer(
        neighborhood_col='Neighborhood', features=neighborhood_feats)),
    ('col_transform', ColumnTransformer([
        ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_pipeline, make_column_selector(dtype_exclude=np.number))
    ], remainder='drop'))
])



# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Create simple pipelines with preprocessor
from sklearn.pipeline import make_pipeline

_prng = np.random.RandomState(42)

y = np.log1p(data['SalePrice'])  # use log-transform for RMSLE
all_X = train_features_df
numeric_cols = all_X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = all_X.select_dtypes(include=['object']).columns.tolist()

models = {
    'LinearRegression': Pipeline([('pre', preprocessor), ('model', LinearRegression())]),
    'Linear_Ridge_default': Pipeline([('pre', preprocessor), ('model', Ridge())]),
    'DecisionTree': Pipeline([('pre', preprocessor), ('model', DecisionTreeRegressor(random_state=42))])
}

# ======================================================
# 4. Define RMSLE metric (RMSE on log scale)
# ======================================================
def rmse_log(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(y_true_log, y_pred_log))


scorer = make_scorer(lambda yt, yp: -rmse_log(yt, yp))  # for cross_val_score

# 5. Cross-validation evaluation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

def check_data_quality(X, y):
    if np.any(pd.isna(X)) or np.any(pd.isna(y)):
        X = X.fillna(0) 
        y = y.fillna(y.median()) if hasattr(y, 'fillna') else y
    return X, y

# Usage:
all_X_clean, y_clean = check_data_quality(all_X.loc[train_features_df.index], y)

for name, pipe in models.items():
    scores = cross_val_score(pipe, all_X.loc[train_features_df.index], y, cv=cv, scoring=scorer, n_jobs=-1)
    rmse_scores = -scores
    cv_results[name] = rmse_scores
    print(f"{name}: RMSE(log) mean={rmse_scores.mean():.4f}, std={rmse_scores.std():.4f}")

print("""Comment: Linear Model works best with ridge (as it removes effect of correlated features 
and gives low CV error, without it linear very overfitted)""")
print("See on the following visualizations, and see the fine tuning to understand")

In [ ]:
# Visualize Important Features and Errors

# Fit models on the full dataset for feature analysis
fitted_models = {}
for name, pipe in models.items():
    try:
        pipe.fit(all_X.loc[data.index], y)
        fitted_models[name] = pipe
        print(f"Successfully fitted {name}")
    except Exception as e:
        print(f"Error fitting {name} for feature analysis: {e}")

warnings.simplefilter("ignore", category=FutureWarning)
# Function to get feature names after transformation
def get_feature_names(column_transformer):
    """Get feature names from all transformers within a ColumnTransformer."""
    output_features = []
    
    # Iterate through transformers in the ColumnTransformer
    for name, pipe, features in column_transformer.transformers_:
        if name == 'remainder' and pipe == 'passthrough':
            # For passthrough, use original feature names (if DataFrame)
             # Adjust based on how remainder features are passed (indices or names)
            if hasattr(column_transformer, '_remainder'):
                 # Scikit-learn >= 0.20 stores remainder info
                 passthrough_features = features[column_transformer._remainder[1]]
                 output_features.extend(passthrough_features)
            continue # Skip further processing for passthrough

        if pipe == 'drop':
             continue # Skip dropped columns
        
        # If the transformer is a Pipeline, get names from the last step (usually encoder)
        if isinstance(pipe, Pipeline):
            final_step = pipe.steps[-1][1]
        else:
            final_step = pipe
            
        # Get feature names from the final step (e.g., OneHotEncoder)
        if hasattr(final_step, 'get_feature_names_out'):
             # Handle features passed to the transformer/pipeline correctly
            if isinstance(features, str): # Single column name
                feature_names = final_step.get_feature_names_out([features])
            elif isinstance(features, slice): # Slice object (less common directly)
                 # Need original column names corresponding to the slice
                 # This requires access to the original DataFrame structure, complex here
                 # Assuming feature indices might be used, but names are preferred
                 print(f"Warning: Slice feature selection for '{name}' complicates name retrieval.")
                 # Attempt to map slice to names if possible, else use generic names
                 try:
                    original_cols = all_X.columns[features] # Requires all_X in scope
                    feature_names = final_step.get_feature_names_out(original_cols)
                 except:
                     feature_names = [f"{name}_slice_{i}" for i in range(len(features))] # Placeholder
            else: # List of column names
                feature_names = final_step.get_feature_names_out(features)

            output_features.extend(list(feature_names))
        else:
             # For transformers without get_feature_names_out (like StandardScaler, SimpleImputer)
             # the feature names remain the same as the input features to that step
             if isinstance(features, str):
                 output_features.append(features)
             elif isinstance(features, list):
                 output_features.extend(features)
             elif hasattr(features, '__iter__'): # Handles selectors/indices if names not directly passed
                # This part can be tricky, might need adjustment based on how features are selected
                # Assuming 'features' holds the column names if it's iterable
                output_features.extend(list(features))


    return output_features


# Analyze Linear Models (Linear Regression and Ridge)
if 'LinearRegression' in fitted_models or 'Linear_Ridge_default' in fitted_models:
    try:
        # Get the ColumnTransformer step from the preprocessor pipeline
        col_transformer_step = fitted_models.get('LinearRegression', fitted_models.get('Linear_Ridge_default'))['pre'].steps[-1][1]
        
        # Get feature names after all transformations
        feature_names = get_feature_names(col_transformer_step)

        # Analyze Linear Regression coefficients
        if 'LinearRegression' in fitted_models:
            lin_reg_model = fitted_models['LinearRegression']['model']
            if hasattr(lin_reg_model, 'coef_'):
                lr_coefs = pd.DataFrame({
                    'Feature': feature_names,
                    'LR_Coefficient': lin_reg_model.coef_
                }).sort_values(by='LR_Coefficient', key=abs, ascending=False)
                
                print("\n--- Linear Regression: Top 10 Coefficients ---")
                display(lr_coefs.head(10))
                
                # Plot coefficients
                plt.figure(figsize=(10, 6))
                sns.barplot(x='LR_Coefficient', y='Feature', data=lr_coefs.head(20), palette='viridis')
                plt.title('Top 20 Feature Coefficients (Linear Regression)')
                plt.tight_layout()
                plt.show()

        # Analyze Ridge coefficients
        if 'Linear_Ridge_default' in fitted_models:
             ridge_model = fitted_models['Linear_Ridge_default']['model']
             if hasattr(ridge_model, 'coef_'):
                 ridge_coefs = pd.DataFrame({
                     'Feature': feature_names,
                     'Ridge_Coefficient': ridge_model.coef_
                 }).sort_values(by='Ridge_Coefficient', key=abs, ascending=False)
                 
                 print("\n--- Ridge Regression: Top 10 Coefficients ---")
                 display(ridge_coefs.head(10))
                 
                 # Plot coefficients
                 plt.figure(figsize=(10, 6))
                 sns.barplot(x='Ridge_Coefficient', y='Feature', data=ridge_coefs.head(20), palette='magma')
                 plt.title('Top 20 Feature Coefficients (Ridge Regression)')
                 plt.tight_layout()
                 plt.show()

    except Exception as e:
        print(f"Error getting feature names or coefficients for linear models: {e}")


# Analyze Decision Tree Feature Importances
if 'DecisionTree' in fitted_models:
    try:
         # Get the ColumnTransformer step from the preprocessor pipeline
        col_transformer_step = fitted_models['DecisionTree']['pre'].steps[-1][1]
        
        # Get feature names after all transformations
        feature_names = get_feature_names(col_transformer_step)

        tree_model = fitted_models['DecisionTree']['model']
        if hasattr(tree_model, 'feature_importances_'):
             tree_importances = pd.DataFrame({
                 'Feature': feature_names,
                 'Importance': tree_model.feature_importances_
             }).sort_values(by='Importance', ascending=False)
             
             print("\n--- Decision Tree: Top 10 Feature Importances ---")
             display(tree_importances.head(10))
             
             # Plot feature importances
             plt.figure(figsize=(10, 6))
             sns.barplot(x='Importance', y='Feature', data=tree_importances.head(20), palette='coolwarm')
             plt.title('Top 20 Feature Importances (Decision Tree)')
             plt.tight_layout()
             plt.show()
             
    except Exception as e:
         print(f"Error getting feature names or importances for Decision Tree: {e}")

if 'Linear_Ridge_default' in fitted_models:
    try:
        ridge_pipe = fitted_models['Linear_Ridge_default']
        y_pred_log = ridge_pipe.predict(all_X.loc[data.index])
        errors_log = y - y_pred_log # Errors on the log scale
        
        plt.figure(figsize=(10, 6))
        sns.histplot(errors_log, kde=True)
        plt.title('Distribution of Prediction Errors (Log Scale) - Ridge')
        plt.xlabel('Log(Actual SalePrice) - Log(Predicted SalePrice)')
        plt.ylabel('Frequency')
        plt.show()
        
        # Scatter plot of actual vs predicted (log scale)
        plt.figure(figsize=(8, 8))
        plt.scatter(y, y_pred_log, alpha=0.5)
        plt.plot([min(y), max(y)], [min(y), max(y)], '--', color='red') # Line y=x
        plt.title('Actual vs. Predicted SalePrice (Log Scale) - Ridge')
        plt.xlabel('Actual Log(SalePrice + 1)')
        plt.ylabel('Predicted Log(SalePrice + 1)')
        plt.grid(True)
        plt.show()

        # Identify largest errors
        error_df = pd.DataFrame({'Actual_Log': y, 'Predicted_Log': y_pred_log, 'Error_Log': errors_log})
        error_df['Abs_Error_Log'] = abs(error_df['Error_Log'])
        largest_errors = error_df.sort_values(by='Abs_Error_Log', ascending=False).head(10)
        
        print("\n--- Examples with Largest Prediction Errors (Log Scale) - Ridge ---")
        # Display original data for houses with largest errors
        display(data.loc[largest_errors.index]) 
        print("\n Corresponding Errors (Log Scale):")
        display(largest_errors[['Actual_Log', 'Predicted_Log', 'Error_Log']])


    except Exception as e:
         print(f"Error during error analysis for Ridge: {e}")

The linear model without ridge coefficients indicate that this model is very overfitted due to the correlation of several of the features. However, with regularization it works great as expected, as Ridge removes effect of correlated features.

Dummy features are very correlated (i.e. if LotShape_Reg = 1, gives indication LotShape_IR1 = 0 for that house)
Other possible correlated features are: 'GrLivArea' is mostly very close to the addition of the first and second floor areas. Therefore not regularized linear works bad. 

Tree models are prone to overfitting and higher generalization error without ensemble techniques etc.

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
# 6. Fine-tune the Models

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import loguniform, randint

print("Starting model fine-tuning...")

# --- Define RMSLE scorer again ---
def rmse_log(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(y_true_log, y_pred_log))
rmsle_scorer = make_scorer(lambda yt, yp: -rmse_log(yt, yp))  # negative for maximization

# 6.1 Fine-tune Ridge regression using RandomizedSearchCV
ridge_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', Ridge(random_state=42))
])

ridge_param_dist = {
    'model__alpha': loguniform(1e-3, 1e3)
}

ridge_search = RandomizedSearchCV(
    ridge_pipe,
    param_distributions=ridge_param_dist,
    n_iter=30,
    cv=5,
    scoring=rmsle_scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

ridge_search.fit(all_X.loc[data.index], y)
print(f"\nBest Ridge params: {ridge_search.best_params_}")
print(f"Best Ridge RMSLE (CV): {-ridge_search.best_score_:.4f}")

# 6.2 Fine-tune Decision Tree using GridSearchCV
tree_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

tree_param_grid = {
    'model__max_depth': [5, 10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [2, 5],
    'model__max_features': ['sqrt', 'log2', None]
}

tree_search = GridSearchCV(
    tree_pipe,
    param_grid=tree_param_grid,
    cv=5,
    scoring=rmsle_scorer,
    n_jobs=-1,
    verbose=1
)

tree_search.fit(all_X.loc[data.index], y)
print(f"\nBest Decision Tree params: {tree_search.best_params_}")
print(f"Best Decision Tree RMSLE (CV): {-tree_search.best_score_:.4f}")

# 6.3 Compare fine-tuned results
tuning_results = pd.DataFrame({
    'Model': ['Linear Regression Ridge (tuned)', 'DecisionTree (tuned)'],
    'Best RMSLE (CV)': [-ridge_search.best_score_, -tree_search.best_score_],
    'Best Params': [ridge_search.best_params_, tree_search.best_params_]
})

display(tuning_results)

print("Fine-tuning complete.")


# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

**1. Non-technical summary (for real-estate executives):**
My Machine Learning solution with predictive model estimates home sale prices based on key property
characteristics such as location, size, and overall quality.
By analyzing past sales data, the model can forecast house prices with
high reliability and low error, helping the company real estate decision-makers make data-driven pricing and investment choices. Key readable plotted visualizations demonstrate great results and predictive strength of my solution.

**2. Technical summary (for the ML team lead):**
I have trained a supervised regression model to predict SalePrice using
engineered numerical, categorical, and ordinal features from the Kaggle
House Prices dataset. The data pipeline handled missing values,
encoded categorical attributes, scaled numeric features, and applied
log transformation to normalize the target variable. Model selection and
tuning were performed using *cross-validation with RMSLE* as the primary metric.
The final model *(Tuned Linear Regressor with Ridge Regularization)* achieved strong predictive accuracy on validation data,
demonstrating *effective generalization* and balanced bias-variance performance.
The tuned linear regressor predicts fast and acheives as low as *0.14 Root Mean Square Log Error and 0.03 standard deviation with cross validation*. Furthermore, I demonstrated through the graphs and visualizations about the predictive robustness of the trained model.

# 9. Submit your predictions to the competition

In [ ]:
test_features_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/test.csv",
        index_col="Id"
    )
)


test_predictions = (
    ridge_search.predict(
        test_features_df
    )
)

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice= np.expm1(test_predictions)
    ).to_csv(
        "submission.csv",
        index=True,
    )
)


In [ ]:
%%bash

cat submission.csv | head -n 5